# Practical Session 02
## Part I - Instructor-led mini-lab
### Example 1 - A small numerical experiment object

We start with a Gaussian wave packet

$$
\psi(x)=\exp\left[-\left(\frac{x-x_0}{\sigma}\right)^2\right]\exp(ik_0x),
\qquad
\rho(x)=|\psi(x)|^2.
$$

Before plotting, think about the following questions:

1. What is the shape of `psi` and `rho`?
2. Which array stores the coordinate information?
3. Which diagnostics should be computed before looking at the figure?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
x = np.linspace(-8.0, 8.0, 2000)

x0 = 0.6
sigma = 1.2
k0 = 8.0

psi = np.exp(-((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
rho = np.abs(psi)**2

print("x.shape:  ", x.shape)
print("psi.shape:", psi.shape)
print("rho.shape:", rho.shape)

assert x.ndim == 1
assert psi.shape == x.shape
assert rho.shape == x.shape
assert np.all(np.diff(x) > 0.0)
assert np.isfinite(rho).all()
assert np.all(rho >= 0.0)

The array contracts above are not only programming checks. They say that the field lives on an ordered one-dimensional grid and that the density is finite and non-negative.

In [ ]:
norm = np.trapezoid(rho, x)

x_mean = np.trapezoid(x * rho, x) / norm
width = np.sqrt(np.trapezoid((x - x_mean)**2 * rho, x) / norm)

imax = np.argmax(rho)
x_peak = x[imax]

diagnostics = {
    "norm": norm,
    "x_mean": x_mean,
    "width": width,
    "x_peak": x_peak,
}

for name, value in diagnostics.items():
    print(f"{name:>10s} = {value:.6g}")

Now use the diagnostics as part of the plot, not as separate information.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.2))

ax.plot(x, rho, label=r"$|\psi(x)|^2$")
ax.axvline(x_mean, linestyle="--", label="centre")

left = x_mean - width
right = x_mean + width
ax.axvspan(left, right, alpha=0.2, label="one width")

ax.plot(x_peak, rho[imax], "o", label="peak")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"density")
ax.set_title("Wave packet density with numerical diagnostics")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

Questions:

1. Does the visual centre agree with `x_mean`?
2. Does `x_peak` agree with the expected location of the maximum?
3. Which part of the plot would become misleading if the `x` array were not passed to `ax.plot`?

## Example 2 - Data, model, residual

A model may look reasonable on top of noisy data while the residual already shows a systematic problem. We generate synthetic data from a known signal and compare them with a slightly imperfect model.

In [ ]:
rng = np.random.default_rng(seed=123)

t = np.linspace(0.0, 10.0, 300)

true_signal = np.exp(-0.25 * t) * np.sin(3.0 * t)
sigma_noise = 0.08

measured = true_signal + sigma_noise * rng.normal(size=t.size)

# A slightly wrong model: the frequency is not exactly correct.
model = np.exp(-0.25 * t) * np.sin(3.12 * t)

residual = measured - model
rms_residual = np.sqrt(np.mean(residual**2))

print(f"RMS residual: {rms_residual:.4f}")
print(f"Expected noise scale: {sigma_noise:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(6.0, 4.2))

axes[0].plot(t, measured, ".", markersize=3, label="synthetic measurement")
axes[0].plot(t, model, linewidth=2, label="model")
axes[0].fill_between(
    t,
    model - sigma_noise,
    model + sigma_noise,
    alpha=0.25,
    label=r"$\pm 1\sigma$",
)
axes[0].set_ylabel(r"signal $s(t)$")
axes[0].legend()

axes[1].axhline(0.0, linewidth=1)
axes[1].plot(t, residual, ".", markersize=3)
axes[1].set_xlabel(r"time $t$ [arb. units]")
axes[1].set_ylabel("residual")
axes[1].set_title(f"RMS residual = {rms_residual:.3f}")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

Questions:

1. Is the residual compatible with unstructured random noise?
2. Which plot makes the wrong frequency easier to notice?
3. Why is the uncertainty band useful even if it is only an approximate model of the noise?

## Example 3 - A two-dimensional field and coordinate conventions

A sampled two-dimensional scalar field has the structure

$$
\rho_{ij}=\rho(x_i,y_j).
$$

We intentionally use an asymmetric Gaussian, because it makes axis mistakes easier to see.

In [ ]:
x2 = np.linspace(-5.0, 5.0, 300)
y2 = np.linspace(-3.0, 3.0, 180)

X2, Y2 = np.meshgrid(x2, y2, indexing="ij")

sigma_x = 1.1
sigma_y = 0.7
x0, y0 = 1.0, -0.4

rho2 = np.exp(
    -(
        (X2 - x0)**2 / (2 * sigma_x**2)
        + (Y2 - y0)**2 / (2 * sigma_y**2)
    )
)

dx2 = x2[1] - x2[0]
dy2 = y2[1] - y2[0]

mass2 = np.sum(rho2) * dx2 * dy2
x_cm = np.sum(X2 * rho2) * dx2 * dy2 / mass2
y_cm = np.sum(Y2 * rho2) * dx2 * dy2 / mass2

imax, jmax = np.unravel_index(np.argmax(rho2), rho2.shape)
x_peak, y_peak = x2[imax], y2[jmax]

print("rho2.shape:", rho2.shape)
print("centre of mass:", x_cm, y_cm)
print("peak location: ", x_peak, y_peak)

First display the raw array. This is useful for debugging storage, but the axes are array indices rather than physical coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 3.4))

im = ax.imshow(rho2)
fig.colorbar(im, ax=ax, label="array value")

ax.set_xlabel("column index")
ax.set_ylabel("row index")
ax.set_title("Raw array display")

fig.tight_layout()

Now plot the same data on the physical grid. The markers should agree with the diagnostic numbers computed above.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))

pcm = ax.pcolormesh(X2, Y2, rho2, shading="auto")

ax.plot(x_peak, y_peak, "o", label="maximum")
ax.plot(x_cm, y_cm, "x", markersize=8, label="centre of mass")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
ax.legend()

fig.colorbar(pcm, ax=ax, label=r"$\rho(x,y)$")
fig.tight_layout()

Questions:

1. Why is the raw `imshow` plot insufficient as a scientific field plot?
2. Where would the peak appear if the axes were accidentally exchanged?
3. Why does the asymmetric choice of `sigma_x`, `sigma_y`, `x0`, and `y0` help with debugging?

## Example 4 - A residual map for a differential equation

For

$$
u(x,y)=\sin(kx)\sin(\ell y),
$$

the continuous equation

$$
u_{xx}+u_{yy}+(k^2+\ell^2)u=0
$$

is satisfied exactly. A finite-difference residual checks how well the sampled field satisfies the equation on the chosen grid.

In [ ]:
x_res = np.linspace(0.0, np.pi, 180)
y_res = np.linspace(0.0, np.pi, 140)

Xr, Yr = np.meshgrid(x_res, y_res, indexing="ij")

k = 2.0
ell = 3.0

u = np.sin(k * Xr) * np.sin(ell * Yr)

dx = x_res[1] - x_res[0]
dy = y_res[1] - y_res[0]

uxx = (u[2:, 1:-1] - 2 * u[1:-1, 1:-1] + u[:-2, 1:-1]) / dx**2
uyy = (u[1:-1, 2:] - 2 * u[1:-1, 1:-1] + u[1:-1, :-2]) / dy**2

R = uxx + uyy + (k**2 + ell**2) * u[1:-1, 1:-1]

Xi = Xr[1:-1, 1:-1]
Yi = Yr[1:-1, 1:-1]

res_inf = np.max(np.abs(R))
res_rms = np.sqrt(np.mean(R**2))

print("u.shape:", u.shape)
print("R.shape:", R.shape)
print(f"||R||_inf = {res_inf:.3e}")
print(f"RMS(R)    = {res_rms:.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))

lim = np.max(np.abs(R))
pcm = ax.pcolormesh(
    Xi,
    Yi,
    R,
    shading="auto",
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
ax.set_title(fr"Residual map, $\|R\|_\infty={lim:.2e}$")

fig.colorbar(pcm, ax=ax, label="residual")
fig.tight_layout()

Questions:

1. Why is the residual array smaller than the original field?
2. Why is a diverging colour scale appropriate here?
3. What additional information does the residual map provide beyond the scalar norm?

Residual maps are not only useful for PDE solvers. Similar visual diagnostics appear in optimization, inverse problems, finite-element methods, and machine learning.

## Part II - Independent work

### Task 1 - Build a diagnostic packet for a wave packet

Use the following setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10.0, 10.0, 2500)

x0 = -1.3
sigma = 1.4
k0 = 6.5

psi = np.exp(-((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
rho = np.abs(psi)**2

Your tasks are:

1. Create a dictionary called `packet` containing at least `x`, `psi`, `rho`, `dx`, and a short text description.
2. Add numerical contracts checking that:
   - `x` is one-dimensional;
   - `psi` and `rho` have the same shape as `x`;
   - `x` is strictly increasing;
   - `rho` is finite and non-negative.
3. Compute the diagnostics:
   - norm;
   - centre;
   - width;
   - peak position;
   - fraction of the norm inside the window
     $$
     |x-\langle x\rangle|<2\,\mathrm{width}.
     $$
4. Store these diagnostics in a second dictionary called `diagnostics`.
5. Make a diagnostic plot showing `rho`, the centre, one-width interval, and the peak.
6. In two or three sentences, explain which diagnostics would immediately reveal a wrong grid, wrong normalization, or wrong axis scaling.

**Optional extension**: Put the whole construction into a function `wave_packet_experiment(x0, sigma, k0, nx)` returning `packet` and `diagnostics`.

In [ ]:
#####################
# Your solution here
#####################

### Task 2 - Debug a misleading two-dimensional field plot

A colleague generated an asymmetric field, but the first plot is difficult to interpret scientifically.

In [ ]:
x = np.linspace(-4.0, 6.0, 260)
y = np.linspace(-3.0, 4.0, 190)

X, Y = np.meshgrid(x, y, indexing="ij")

field = np.exp(
    -(
        (X - 2.0)**2 / (2 * 0.8**2)
        + (Y + 1.0)**2 / (2 * 1.4**2)
    )
)

# One artificial outlier that should not silently control the whole colour scale.
field_with_outlier = field.copy()
field_with_outlier[25, 150] = 12.0 * field.max()

fig, ax = plt.subplots(figsize=(5.0, 3.4))
im = ax.imshow(field_with_outlier)
fig.colorbar(im, ax=ax)
ax.set_title("First attempt")
plt.show()

Your tasks are:

1. Compute the integral, centre of mass, and maximum location of `field` without the outlier.
2. Make a corrected field plot using physical coordinates. You may use either:
   - `pcolormesh(X, Y, field, shading="auto")`, or
   - `imshow(field.T, extent=[...], origin="lower", aspect="auto")`.
3. Overlay the centre of mass and maximum location on the corrected plot.
4. Set an aspect ratio that preserves the geometry of the field.
5. Make a second plot of `field_with_outlier`, but choose colour limits consciously using the 1st and 99th percentiles.
6. Label both coordinate axes and the colourbar.
7. Conclude with two or three sentences explaining how default plotting choices can hide or invent apparent structure.

**Optional extension**: Compare `imshow` and `pcolormesh` for this example. Which version makes the coordinate convention easier to see?

In [ ]:
#####################
# Your solution here
#####################

### Task 3 - Residual maps and convergence as a visual numerical experiment

For

$$
u(x,y)=\sin(kx)\sin(\ell y),
$$

the continuous equation

$$
u_{xx}+u_{yy}+(k^2+\ell^2)u=0
$$

is satisfied exactly. The discrete residual should decrease when the grid is refined.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 2.0
ell = 3.0

def make_field(nx, ny):
    x = np.linspace(0.0, np.pi, nx)
    y = np.linspace(0.0, np.pi, ny)
    X, Y = np.meshgrid(x, y, indexing="ij")
    u = np.sin(k * X) * np.sin(ell * Y)
    return x, y, X, Y, u

Your tasks are:

1. Write a function `residual_helmholtz(x, y, u, k, ell)` that returns:
   - the interior coordinate arrays;
   - the residual array;
   - the maximum norm of the residual;
   - the RMS norm of the residual.
2. Use centred finite differences on the interior grid points.
3. Check explicitly that the residual shape is `(nx-2, ny-2)`.
4. For one medium-sized grid, for example `(nx, ny) = (180, 140)`, make:
   - a plot of the field `u`;
   - a signed residual map with a centred diverging colour scale.
5. Repeat the calculation for several grid sizes, for example:
   ```python
   sizes = [(40, 32), (80, 64), (160, 128), (320, 256)]
   ```
6. Store the grid spacing and residual norms in arrays.
7. Make a log-log convergence plot of the maximum residual norm versus grid spacing.
8. Add a reference line proportional to $\Delta x^2$.
9. Estimate the observed order from neighbouring error ratios.
10. Write a short interpretation: does the visual convergence agree with the expected second-order finite-difference stencil?

**Optional extension**: Repeat the test for a higher wavenumber, for example `k = 5`, `ell = 7`. What changes: the order, the error constant, or both?

In [ ]:
#####################
# Your solution here
#####################